In [25]:
import polars as pl
import pandas as pd
import glob

# SII

In [26]:
COLS_SII_UTILIZADAS = [
    "Tipo Doc",
    "RUT Emisor",
    "Razon Social",
    "Folio",
    "Fecha Docto",
    "Fecha Recepcion",
    "Fecha Reclamo",
    "Fecha Acuse" "Monto Exento",
    "Monto Neto",
    "Monto IVA Recuperable",
    "Monto Total",
]

schema = {
    "Nro": pl.Int64,
    "Tipo Doc": pl.Int64,
    "Tipo Compra": pl.Utf8,
    "RUT Proveedor": pl.Utf8,
    "Razon Social": pl.Utf8,
    "Folio": pl.Int64,
    "Fecha Docto": pl.Utf8,
    "Fecha Recepcion": pl.Utf8,
    "Fecha Acuse": pl.Utf8,
    "Monto Exento": pl.Int64,
    "Monto Neto": pl.Int64,
    "Monto IVA Recuperable": pl.Int64,
    "Monto Iva No Recuperable": pl.Int64,
    "Codigo IVA No Rec.": pl.Int64,
    "Monto Total": pl.Int64,
    "Monto Neto Activo Fijo": pl.Int64,
    "IVA Activo Fijo": pl.Int64,
    "IVA uso Comun": pl.Int64,
    "Impto. Sin Derecho a Credito": pl.Int64,
    "IVA No Retenido": pl.Int64,
    "NCE o NDE sobre Fact. de Compra": pl.Int64,
    "Codigo Otro Impuesto": pl.Utf8,
    "Valor Otro Impuesto": pl.Utf8,
    "Tasa Otro Impuesto": pl.Utf8,
}

df = (
    pl.scan_csv("crudos/base_de_datos_facturas/SII/*.csv", separator=";", dtypes=schema)
    .rename({"RUT Proveedor": "RUT Emisor"})
    .with_columns(
        pl.when((pl.col("Tipo Doc") == 61) | (pl.col("Tipo Doc") == 56)).then(
            pl.col(["Monto Exento", "Monto Neto", "Monto IVA Recuperable", "Monto Total"]) * -1
        )
    )
    .with_columns(pl.concat_str(pl.col(["RUT Emisor", "Folio"])).alias("llave_id"))
)

# ACEPTA

In [27]:
lista_acepta = glob.glob("crudos/base_de_datos_facturas/ACEPTA/*.xls")
df = map(pd.read_excel, lista_acepta)
df_sumada = pd.concat(df)

# SCI

In [28]:
df = pl.scan_csv(
    "crudos/base_de_datos_facturas/SCI/*.csv", separator=",", dtypes={"Numero Documento": pl.Utf8}
).rename({"Rut Proveedor": "RUT Emisor", "Numero Documento": "Folio"})

# SIGFE REPORTS - Facturas

In [51]:
df = (
    pl.scan_csv("crudos/base_de_datos_facturas/SIGFE/*.csv", separator=",", skip_rows=10)
    .filter((pl.col("Folio").is_not_null()))
    .filter((pl.col("Cuenta Contable") != "Cuenta Contable"))
    .with_columns(
        [
            pl.col("Principal")
            .str.split(" ")
            .arr[0]
            .str.replace(".", " ", literal=True)
            .str.to_uppercase()
            .str.strip()
            .alias("RUT Emisor"),
            pl.col("Fecha").str.strptime(pl.Date, fmt="%d/%m/%Y").cast(pl.Date),
            pl.col("Folio").cast(pl.Int32),
        ]
    )
    .rename({"Folio": "Folio_interno", "Número ": "Folio"})
)